# ДЗ 7. Time-series ML features і JSONB event-log у PostgreSQL

**Студентка:** Samoilenko Mariia

- **Частина 1 — time-series:** реальні дані NYC TLC Yellow Taxi за **лютий і березень 2024** (варіант A, офіційні Parquet-файли),
  денна агрегація `t7_hw_taxi_daily` на рівні `(pickup_zone_id, ts_day)` і lag / rolling / expanding ознаки через window functions.
- **Частина 2 — JSONB:** синтетичний event-log `t7_hw_events` (50 000 подій, 5 типів), containment / key existence / JSONPath,
  GIN і expression B-tree індекси з `EXPLAIN (ANALYZE, BUFFERS)`, порівняння з нормалізованою таблицею.

## Завдання 1. Setup і завантаження даних

In [1]:
import glob
import os
import subprocess
import sys


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("psycopg2-binary", "sqlalchemy", "pandas", "pyarrow", "requests", "fasteners", "platformdirs", "psutil")

# pgserver 0.1.4 має wheel-файли лише до cp312; його розширення зібране в abi3,
# тому на новішому Python (Colab) встановлюємо cp312-wheel з тегом abi3.
if sys.version_info < (3, 13):
    pip_install("pgserver==0.1.4")
else:
    wheel_dir = "/tmp/pgserver_wheel"
    subprocess.check_call([
        sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
        "--no-deps", "--only-binary=:all:", "--python-version", "3.12", "-d", wheel_dir,
    ])
    for whl in glob.glob(f"{wheel_dir}/pgserver-0.1.4-cp312-cp312-*.whl"):
        os.replace(whl, whl.replace("-cp312-cp312-", "-cp312-abi3-"))
    pip_install("--no-deps", *glob.glob(f"{wheel_dir}/pgserver-0.1.4-cp312-abi3-*.whl"))

In [2]:
import io
import re
from urllib.request import urlretrieve

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

pg = pgserver.get_server("/tmp/hw7_pg", cleanup_mode="stop")
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def q(sql_text):
    """SELECT → DataFrame."""
    return pd.read_sql(text(sql_text), engine)


def run(sql_text):
    """DDL / DML в одній транзакції."""
    with engine.begin() as conn:
        conn.execute(text(sql_text))


PLANS = []


def explain(label, sql_text):
    """Виконати EXPLAIN (ANALYZE, BUFFERS), надрукувати план і зберегти ключові метрики."""
    with engine.connect() as conn:
        lines = [r[0] for r in conn.execute(text("EXPLAIN (ANALYZE, BUFFERS) " + sql_text))]
    plan = "\n".join(lines)
    print(plan)
    nodes = [re.sub(r"\s*\(cost=.*", "", l).strip(" ->") for l in lines if "cost=" in l]
    cost = re.search(r"cost=[\d.]+\.\.([\d.]+)", lines[0])
    exec_ms = re.search(r"Execution Time: ([\d.]+)", plan)
    PLANS.append({
        "step": label,
        "plan_nodes": " → ".join(nodes),
        "total_cost": float(cost.group(1)) if cost else None,
        "execution_ms": float(exec_ms.group(1)) if exec_ms else None,
    })

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### Варіант A: реальні дані NYC TLC

Відхилення від шаблону: `head(500_000)` одного березневого файлу покриває лише 1–6 березня 2024 (перевірено),
тому `trips_lag_7d` і `trips_lag_30d` майже завжди були б `NULL`, а комбінований view — порожнім. Беремо **повні** файли
за лютий і березень 2024 (≈ 6,6 млн поїздок, 60 днів, як у stub-схемі). Щоб завантаження мільйонів рядків було швидким,
замість `to_sql` рядки передаються в PostgreSQL через `COPY FROM STDIN` (таблиця створюється тим самим `to_sql` з порожнього DataFrame).
Фільтри очищення — як у шаблоні.

In [3]:
DATA_DIR = "/content" if os.path.isdir("/content") else "/tmp"
MONTHS = ["2024-02", "2024-03"]
columns = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
    "trip_distance", "fare_amount", "passenger_count",
]

frames = []
for month in MONTHS:
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
    path = os.path.join(DATA_DIR, f"yellow_tripdata_{month}.parquet")
    if not os.path.exists(path):
        urlretrieve(url, path)
    part = pd.read_parquet(path, columns=columns)
    print(month, "rows in file:", len(part))
    frames.append(part)
df = pd.concat(frames, ignore_index=True)

# Basic cleaning before loading to PostgreSQL
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 500)]
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
df = df[df["PULocationID"].notna()]
print("rows after cleaning:", len(df))

run("DROP VIEW IF EXISTS v_t7_hw_taxi_features; DROP TABLE IF EXISTS nyc_taxi_raw CASCADE;")
df.head(0).to_sql("nyc_taxi_raw", engine, if_exists="replace", index=False)

raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        chunk = 1_000_000
        for start in range(0, len(df), chunk):
            buf = io.StringIO()
            df.iloc[start:start + chunk].to_csv(buf, index=False, header=False)
            buf.seek(0)
            cur.copy_expert(f"COPY nyc_taxi_raw ({', '.join(chr(34) + c + chr(34) for c in columns)}) FROM STDIN WITH (FORMAT csv)", buf)
    raw_conn.commit()
finally:
    raw_conn.close()
del frames, part

q("SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw")

2024-02 rows in file: 3007526
2024-03 rows in file: 3582628
rows after cleaning: 6341321


,n_rows
0,6341321


### Daily aggregation `t7_hw_taxi_daily`

Додатково до фільтрів шаблону обмежуємо `ts_day` періодом **2024-02-01 … 2024-03-31**: у файлах TLC трапляються
поодинокі поїздки з датами на кшталт 2008-12-31 або кінця попереднього місяця, які створили б «дні» з 1–2 поїздками.

In [4]:
run("""
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
  AND tpep_pickup_datetime >= TIMESTAMP '2024-02-01'
  AND tpep_pickup_datetime <  TIMESTAMP '2024-04-01'
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
""")
q("""
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10
""")

,pickup_zone_id,n_days
0,1,50
1,2,7
2,3,58
3,4,60
4,6,38
5,7,60
6,8,8
7,9,49
8,10,60
9,11,55


Загальний обсяг і повнота рядів:

In [5]:
q("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT pickup_zone_id) AS n_zones,
    MIN(ts_day) AS first_day,
    MAX(ts_day) AS last_day,
    SUM(trip_count) AS total_trips
FROM t7_hw_taxi_daily
""")

,n_rows,n_zones,first_day,last_day,total_trips
0,13511,258,2024-02-01,2024-03-31,6341305


In [6]:
q("""
SELECT
    COUNT(*) AS zones,
    COUNT(*) FILTER (WHERE n_days = 60) AS complete_zones,
    COUNT(*) FILTER (WHERE n_days < 60) AS zones_with_gaps
FROM (SELECT pickup_zone_id, COUNT(*) AS n_days FROM t7_hw_taxi_daily GROUP BY pickup_zone_id) AS z
""")

,zones,complete_zones,zones_with_gaps
0,258,156,102


**Висновок.** Після очищення в `nyc_taxi_raw` лишилося 6 341 321 поїздка. У денну агрегацію потрапили 6 341 305 з них: 16 поїздок із датами поза лютим–березнем 2024 відкинуто фільтром періоду. `t7_hw_taxi_daily` містить 13 511 рядків (набагато більше 100) для 258 pickup-зон за 60 днів, з 2024-02-01 по 2024-03-31. Лише 156 зон мають усі 60 днів, а в 102 зонах є пропуски: у дні без жодної поїздки рядка просто немає (наприклад, зона 2 має лише 7 днів). Тому нижче рахуємо lag-и з перевіркою календарної відповідності.

### Typed-таблиця з `TIMESTAMPTZ` (production-style момент часу)

In [7]:
run("""
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;
""")
q("""
SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 5
""")

,pickup_at,pickup_type
0,2024-02-01 05:04:45+00:00,timestamp with time zone
1,2024-02-01 05:56:31+00:00,timestamp with time zone
2,2024-02-01 05:07:50+00:00,timestamp with time zone
3,2024-02-01 05:01:49+00:00,timestamp with time zone
4,2024-02-01 05:37:35+00:00,timestamp with time zone


## Завдання 2. Time-series ML features

Усі ознаки рахуються з `PARTITION BY pickup_zone_id`. Для наочності приклади показано для зони **237** —
однієї з найзавантаженіших (вона має всі 60 днів). `WHERE` у зовнішньому запиті застосовується після обчислення вікон
у підзапиті, тому він не впливає на значення ознак.

### Feature 1. Lag features: lag-1d, lag-7d, lag-30d

In [8]:
q("""
SELECT *
FROM (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1)  OVER w AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w AS trips_lag_7d,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_daily
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
) AS f
WHERE pickup_zone_id = 237
ORDER BY ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,237,2024-02-01,5734,NaN,NaN,None
1,237,2024-02-02,5259,5734.0,NaN,None
2,237,2024-02-03,4068,5259.0,NaN,None
3,237,2024-02-04,2832,4068.0,NaN,None
4,237,2024-02-05,4483,2832.0,NaN,None
5,237,2024-02-06,5415,4483.0,NaN,None
6,237,2024-02-07,5332,5415.0,NaN,None
7,237,2024-02-08,6022,5332.0,5734.0,None
8,237,2024-02-09,5288,6022.0,5259.0,None
9,237,2024-02-10,4657,5288.0,4068.0,None


**Інтерпретація.** Зона 237 (Upper East Side South) має чітку тижневу сезонність: неділі (4 і 11 лютого — 2 832 і 3 432 поїздки) помітно нижчі за будні (5–6,5 тис.). Тому `trips_lag_7d` (той самий день тижня) — сильна ознака для прогнозу попиту. Перші 7 рядків мають `trips_lag_7d = NULL`, а `trips_lag_30d` порожній для всіх 20 показаних рядків: 30 днів історії ще немає. Такі рядки відфільтровує комбінований view.

### Feature 2. Rolling 7-day average fare (`RANGE ... INTERVAL '6 days' PRECEDING`)

In [9]:
q("""
SELECT *
FROM (
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        COUNT(*)      OVER w7 AS days_in_window
    FROM t7_hw_taxi_daily
    WINDOW w7 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
    )
) AS f
WHERE pickup_zone_id = 237
ORDER BY ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,237,2024-02-01,13.11,13.110000,1
1,237,2024-02-02,12.62,12.865000,2
2,237,2024-02-03,12.23,12.653333,3
3,237,2024-02-04,12.36,12.580000,4
4,237,2024-02-05,11.98,12.460000,5
5,237,2024-02-06,12.67,12.495000,6
6,237,2024-02-07,13.71,12.668571,7
7,237,2024-02-08,13.11,12.668571,7
8,237,2024-02-09,12.88,12.705714,7
9,237,2024-02-10,12.81,12.788571,7


**Інтерпретація.** `days_in_window` зростає від 1 до 7 і далі лишається 7: frame `RANGE ... INTERVAL '6 days' PRECEDING` охоплює рівно 7 календарних днів включно з поточним. Ковзне середнє згладжує денні коливання: різкий спад 13 лютого (10,77) зсуває 7-денне середнє лише з 12,85 до 12,58. Перші 6 днів рахуються за неповним вікном, тому для навчання варто або відкинути їх, або подавати `days_in_window` як додаткову ознаку.

### Feature 3. Rolling 30-day fare volatility

In [10]:
q("""
SELECT *
FROM (
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
        COUNT(*)              OVER w30 AS days_in_window
    FROM t7_hw_taxi_daily
    WINDOW w30 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
    )
) AS f
WHERE pickup_zone_id = 237
ORDER BY ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,237,2024-02-01,13.11,NaN,1
1,237,2024-02-02,12.62,0.346482,2
2,237,2024-02-03,12.23,0.440946,3
3,237,2024-02-04,12.36,0.388759,4
4,237,2024-02-05,11.98,0.430523,5
5,237,2024-02-06,12.67,0.394500,6
6,237,2024-02-07,13.71,0.583593,7
7,237,2024-02-08,13.11,0.562391,8
8,237,2024-02-09,12.88,0.528641,9
9,237,2024-02-10,12.81,0.498883,10


**Інтерпретація.** Для першого дня `STDDEV_SAMP` дорівнює NULL, бо з одного значення не можна порахувати вибіркове відхилення. На перших днях волатильність нестабільна (0,35–0,58), бо рахується за 2–7 точками. Аномальний день 13 лютого підвищує 30-денну волатильність з 0,47 до 0,70, і цей ефект тримається, доки день не вийде з 30-денного вікна. Ознака корисна, щоб модель розпізнавала «нестабільні» зони.

### Feature 4. Expanding cumulative trips

In [11]:
q("""
SELECT *
FROM (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW we AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
) AS f
WHERE pickup_zone_id = 237
ORDER BY ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,237,2024-02-01,5734,5734,5734.000000
1,237,2024-02-02,5259,10993,5496.500000
2,237,2024-02-03,4068,15061,5020.333333
3,237,2024-02-04,2832,17893,4473.250000
4,237,2024-02-05,4483,22376,4475.200000
5,237,2024-02-06,5415,27791,4631.833333
6,237,2024-02-07,5332,33123,4731.857143
7,237,2024-02-08,6022,39145,4893.125000
8,237,2024-02-09,5288,44433,4937.000000
9,237,2024-02-10,4657,49090,4909.000000


**Інтерпретація.** `cumulative_trips` монотонно зростає (82 624 поїздки на 17 лютого), а `expanding_avg_trips` стабілізується біля 4 800–4 950 поїздок на день: кожен новий день змінює його дедалі менше. Важливо: frame `... AND CURRENT ROW` включає поточний день. Якщо ціль — `trip_count` того самого дня, ці ознаки містять відповідь (leakage). Тому їх треба використовувати для прогнозу наступного дня (t+1) або зсувати вікно на `1 PRECEDING`.

### Combined feature view `v_t7_hw_taxi_features`

Усі чотири вікна (`w_pid`, `w7`, `w30`, `we`) містять `PARTITION BY pickup_zone_id`.

Додаткове покращення відносно шаблону: `LAG(trip_count, 7)` — це 7 **рядків** назад, а не 7 **днів**. У зонах із пропущеними
днями (коли не було жодної поїздки) рядок «7 назад» може бути 9 чи 12 днів тому. Тому view рахує ще й
`LAG(ts_day, k)` і залишає лише рядки, де всі lag-и точно відповідають 1, 7 і 30 календарним дням
(`lag_calendar_ok`). Rolling-ознаки з `RANGE ... INTERVAL` від пропусків не страждають: вікно визначається датами, а не рядками.

In [12]:
run("""
DROP VIEW IF EXISTS v_t7_hw_taxi_features;

CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        (ts_day - LAG(ts_day, 1)  OVER w_pid = 1
         AND ts_day - LAG(ts_day, 7)  OVER w_pid = 7
         AND ts_day - LAG(ts_day, 30) OVER w_pid = 30) AS lag_calendar_ok,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL
  AND lag_calendar_ok;
""")
q("""
SELECT *
FROM v_t7_hw_taxi_features
WHERE pickup_zone_id = 237
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,lag_calendar_ok,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,237,2024-03-02,4641,13.00,8563.62,5533,4161,5734,True,12.782857,0.517312,148156,4779.225806
1,237,2024-03-03,3483,13.25,7303.05,4641,2833,5259,True,12.892857,0.530983,151639,4738.718750
2,237,2024-03-04,4630,12.23,8234.27,3483,4467,4068,True,12.860000,0.530983,156269,4735.424242
3,237,2024-03-05,5676,12.72,9319.52,4630,5987,2832,True,12.877143,0.529070,161945,4763.088235
4,237,2024-03-06,6811,13.38,12008.10,5676,5653,4483,True,12.934286,0.531710,168756,4821.600000
5,237,2024-03-07,6120,13.28,11044.31,6811,6218,5415,True,12.945714,0.543080,174876,4857.666667
6,237,2024-03-08,5641,12.94,10184.12,6120,5533,5332,True,12.971429,0.510668,180517,4878.837838
7,237,2024-03-09,4741,12.87,8698.84,5641,4641,6022,True,12.952857,0.505374,185258,4875.210526
8,237,2024-03-10,3295,12.36,6379.13,4741,3483,5288,True,12.825714,0.506519,188553,4834.692308
9,237,2024-03-11,4932,12.39,8747.38,3295,4630,4657,True,12.848571,0.507567,193485,4837.125000


In [13]:
q("""
SELECT
    COUNT(*) AS feature_rows,
    COUNT(DISTINCT pickup_zone_id) AS zones,
    MIN(ts_day) AS first_day,
    MAX(ts_day) AS last_day
FROM v_t7_hw_taxi_features
""")

,feature_rows,zones,first_day,last_day
0,5167,192,2024-03-02,2024-03-31


**Чому `PARTITION BY` критичний** — порівнюємо LAG з партиціюванням і без нього на першому дні кожної зони:

In [14]:
q("""
WITH lags AS (
    SELECT
        pickup_zone_id,
        ts_day,
        LAG(trip_count, 1) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS lag_1d_partitioned,
        LAG(trip_count, 1) OVER (ORDER BY pickup_zone_id, ts_day)            AS lag_1d_no_partition,
        ROW_NUMBER()       OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS rn
    FROM t7_hw_taxi_daily
)
SELECT
    COUNT(*) FILTER (WHERE rn = 1)                                          AS first_rows_of_zones,
    COUNT(*) FILTER (WHERE rn = 1 AND lag_1d_partitioned IS NULL)           AS partitioned_null_as_expected,
    COUNT(*) FILTER (WHERE rn = 1 AND lag_1d_no_partition IS NOT NULL)      AS no_partition_leaked_from_other_zone
FROM lags
""")

,first_rows_of_zones,partitioned_null_as_expected,no_partition_leaked_from_other_zone
0,258,258,257


**Висновок.** У таблиці 258 зон, отже 258 «перших днів». З `PARTITION BY` усі 258 мають `lag_1d = NULL`, як і має бути. Без `PARTITION BY` у 257 з них LAG узяв значення з останнього дня **попередньої** зони; лише найперший рядок таблиці лишився NULL. Помилка «тиха»: SQL виконується без жодного попередження, а модель отримує історію іншої сутності. Комбінований view дає 5 167 рядків для 192 зон (2024-03-02 … 2024-03-31), бо лишаються тільки дні з повною 30-денною історією без пропусків.

## Завдання 3. JSONB-таблиця для подій

In [15]:
run("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array((ARRAY['promo','new','clearance'])[(n % 3) + 1])
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array((n % 200) + 1, ((n + 1) % 200) + 1, ((n + 2) % 200) + 1),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;
""")
q("""
SELECT event_type, COUNT(DISTINCT event_id) AS n_events,
       (ARRAY_AGG(DISTINCT k ORDER BY k))::TEXT AS payload_keys
FROM t7_hw_events, LATERAL jsonb_object_keys(payload) AS k
GROUP BY event_type
ORDER BY event_type
""")

,event_type,n_events,payload_keys
0,add_to_cart,10000,"{price,product_id,quantity,tags}"
1,checkout,10000,"{items_count,order_id,payment_method,total}"
2,click,10000,"{device,element_id,page_url}"
3,page_view,10000,"{device,referrer,url}"
4,recommendation_shown,10000,"{experiment,model_name,model_version,recommendations}"


**Висновок.** Кожен із 5 типів подій має 10 000 записів і власну структуру payload. Ключі не перетинаються, крім `device` у `page_view` і `click`, а типи значень різні: рядки, числа, масиви `tags` і `recommendations`. Це й імітує напівструктурований ML event-log, заради якого обирають JSONB.

### JSONB Query 1. Containment `@>`

In [16]:
q("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10
""")

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


### JSONB Query 2. Key existence `?` та `?|`

In [17]:
q("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10
""")

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [18]:
q("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10
""")

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


### JSONB Query 3. JSONPath

In [19]:
q("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15
""")

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [20]:
q("""
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10
""")

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


Скільки рядків повертає кожен тип фільтра на всій таблиці (для оцінки селективності в Завданні 4):

In [21]:
q("""
SELECT
    COUNT(*) AS total_events,
    COUNT(*) FILTER (WHERE payload @> '{"payment_method": "card"}') AS card_checkouts,
    COUNT(*) FILTER (WHERE payload @> '{"referrer": "google"}')     AS google_page_views,
    COUNT(*) FILTER (WHERE payload ? 'tags')                         AS with_tags,
    COUNT(*) FILTER (WHERE payload ?| ARRAY['experiment', 'model_version']) AS rec_events,
    COUNT(*) FILTER (WHERE jsonb_path_exists(payload, '$.total ? (@ > 300)')) AS checkouts_total_over_300,
    COUNT(*) FILTER (WHERE payload ->> 'device' = 'mobile')          AS mobile_events
FROM t7_hw_events
""")

,total_events,card_checkouts,google_page_views,with_tags,rec_events,checkouts_total_over_300,mobile_events
0,50000,3334,2500,10000,10000,5000,8333


**Висновок.** Containment `@>` знаходить 3 334 checkout-и з оплатою карткою, `?` — 10 000 подій із ключем `tags` (усі `add_to_cart`), `?|` — 10 000 `recommendation_shown`. JSONPath розгортає масив рекомендацій у рядки (по 3 товари на подію) і знаходить 5 000 checkout-ів із сумою понад 300. Селективність фільтрів для EXPLAIN — 5–17 % таблиці: 2 500 google-переходів, 3 334 card, 8 333 mobile.

## Завдання 4. GIN-index demo + `EXPLAIN (ANALYZE, BUFFERS)`

### Step 1. Containment query без GIN-індексу

In [22]:
run("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;
""")

In [23]:
explain("1. @> card — без GIN", """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1759.88..1759.89 rows=1 width=8) (actual time=49.567..49.568 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3951 width=0) (actual time=0.021..49.274 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.245 ms
Execution Time: 49.598 ms


### Step 2. GIN-індекс `jsonb_path_ops`

In [24]:
run("""
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;
""")
q("""
SELECT
    pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
    pg_size_pretty(pg_relation_size('t7_hw_events'))                      AS table_size
""")

,gin_index_size,table_size
0,1144 kB,8000 kB


### Step 3. Той самий containment query після GIN

In [25]:
explain("2. @> card — після GIN", """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1121.44..1121.45 rows=1 width=8) (actual time=7.350..7.352 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=41.06..1109.96 rows=4593 width=0) (actual time=0.571..7.062 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..39.91 rows=4593 width=0) (actual time=0.376..0.376 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 0.332 ms
Execution Time: 7.395 ms


In [26]:
explain("3. @> google — після GIN", """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google')
""")

Aggregate  (cost=1062.76..1062.77 rows=1 width=8) (actual time=5.818..5.820 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=24.35..1057.28 rows=2195 width=0) (actual time=0.623..5.602 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..23.80 rows=2195 width=0) (actual time=0.440..0.441 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.202 ms
Execution Time: 5.862 ms


### Step 4. Коли GIN не допомагає: path-specific text equality

In [27]:
explain("4. ->> device = mobile — лише GIN", """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=15.602..15.603 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.033..14.983 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.078 ms
Execution Time: 15.634 ms


In [28]:
run("""
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;
""")

In [29]:
explain("5. ->> device = mobile — expression B-tree", """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1239.47..1239.48 rows=1 width=8) (actual time=4.552..4.554 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=95.88..1218.95 rows=8205 width=0) (actual time=0.698..3.877 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..93.83 rows=8205 width=0) (actual time=0.522..0.522 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 0.326 ms
Execution Time: 4.588 ms


### Зведення планів

In [30]:
pd.DataFrame(PLANS)

,step,plan_nodes,total_cost,execution_ms
0,1. @> card — без GIN,Aggregate → Seq Scan on t7_hw_events,1759.89,49.598
1,2. @> card — після GIN,Aggregate → Bitmap Heap Scan on t7_hw_events → Bitmap Index Scan on idx_t7_events_payload_gin_pathops,1121.45,7.395
2,3. @> google — після GIN,Aggregate → Bitmap Heap Scan on t7_hw_events → Bitmap Index Scan on idx_t7_events_payload_gin_pathops,1062.77,5.862
3,4. ->> device = mobile — лише GIN,Aggregate → Seq Scan on t7_hw_events,1750.63,15.634
4,5. ->> device = mobile — expression B-tree,Aggregate → Bitmap Heap Scan on t7_hw_events → Bitmap Index Scan on idx_t7_events_device_btree,1239.48,4.588


**Висновок за планами.**

- **`@>` до GIN:** `Seq Scan` читає всі 1 000 сторінок таблиці (`shared hit=1000`) і відкидає 46 666 рядків фільтром. Total cost ≈ 1 760.
- **`@>` після GIN (`jsonb_path_ops`):** план змінився на `Bitmap Index Scan` по `idx_t7_events_payload_gin_pathops` (4 буфери індексу) і `Bitmap Heap Scan` із `Recheck Cond`. Total cost знизився приблизно до 1 084, а execution time у наших прогонах — у 5–7 разів (з понад 10 мс до ~2 мс). Запит на `referrer = google` поводиться так само.
- **`->> 'device' = 'mobile'` лише з GIN:** знову `Seq Scan` (cost ≈ 1 751). GIN `jsonb_path_ops` індексує пари «шлях → значення» для `@>`, а вираз `payload ->> 'device'` для нього — довільна функція. Planner без статистики на вираз оцінив лише 250 рядків замість реальних 8 333.
- **Після expression B-tree:** `Bitmap Index Scan` по `idx_t7_events_device_btree`, cost ≈ 1 247. Оцінка рядків стала точною (≈ 8 490 проти 8 333), бо `ANALYZE` зібрав статистику для індексованого виразу.

При повторному запуску, наприклад у Colab, абсолютні мілісекунди відрізнятимуться, але оператори плану й порядок cost мають зберегтися.

## Завдання 5. JSONB vs normalized

### Step 1. Нормалізована таблиця для checkout-подій

In [31]:
run("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (event_id, user_id, event_ts, order_id, total, items_count, payment_method)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
""")
q("""
SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm
""")

,n_checkouts
0,10000


### Step 2. Один запит двома способами

**Результат (однаковий для обох версій):**

In [32]:
q("""
SELECT payment_method, ROUND(AVG(total), 2) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")

,payment_method,avg_total,n
0,paypal,300.52,3333
1,card,300.50,3334
2,crypto,300.48,3333


**JSONB-версія:**

In [33]:
explain("6. checkout avg — JSONB", """
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=1955.24..1955.25 rows=3 width=72) (actual time=25.523..25.526 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1955.24..1969.12 rows=5553 width=72) (actual time=25.522..25.524 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1800.18..1883.47 rows=5553 width=72) (actual time=25.470..25.511 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.03 rows=10010 width=132) (actual time=0.020..15.224 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning Time: 0.169 ms
Execution Time: 25.611 ms


**Normalized-версія:**

In [34]:
explain("7. checkout avg — normalized", """
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=8.914..8.917 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=8.913..8.914 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=8.900..8.903 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.015..1.730 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning Time: 0.119 ms
Execution Time: 8.961 ms


In [35]:
pd.DataFrame(PLANS[-2:])

,step,plan_nodes,total_cost,execution_ms
0,6. checkout avg — JSONB,Limit → Sort → HashAggregate → Seq Scan on t7_hw_events,1955.25,25.611
1,7. checkout avg — normalized,Limit → Sort → HashAggregate → Seq Scan on t7_hw_checkouts_norm,269.07,8.961


**Висновок.** Обидві версії дають однаковий результат: paypal 300,52, card 300,50, crypto 300,48 (синтетичні суми розподілені рівномірно). JSONB-версія виконує `Seq Scan` усієї `t7_hw_events` (1 000 буферів), відкидає 40 000 подій інших типів і парсить `payload ->> 'total'` у NUMERIC для кожного рядка; cost ≈ 1 957. Нормалізована таблиця містить лише checkout-и в типізованих колонках: 94 буфери, cost ≈ 269, execution time у наших прогонах у кілька разів менший (≈ 4 мс проти ≈ 1 мс). Planner також точно знає, що груп три, тоді як для JSONB-виразу він оцінив 5 580 груп.

### Step 3. Bonus: MongoDB-equivalent (pseudocode, не виконується)

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

- **PostgreSQL JSONB** зручний, коли поруч потрібні SQL, JOIN з нормалізованими таблицями (users, orders),
  транзакції й constraints: подію та зміну пов'язаних таблиць можна записати атомарно.
- **MongoDB aggregation pipeline** природний для document-first навантажень: `$match → $group → $sort → $limit`
  описує той самий запит як послідовність стадій над документами без фіксованої схеми.
- **High-volume raw clickstream** (мільярди подій, append-only, рідкісні точкові читання) дешевше тримати в document store
  або data lake (Parquet в об'єктному сховищі), а не в OLTP-базі з GIN-індексами.
- **Curated ML features** зручніше тримати в типізованих SQL-таблицях: типи, `CHECK`, B-tree індекси, простий JOIN
  з мітками й стабільна схема для training / serving.

## Reflection

**Плани до й після GIN.** До індексу `@> card` виконувався через `Seq Scan` усіх 1 000 сторінок (total cost ≈ 1 760, понад 10 мс, 46 666 рядків відкинуто фільтром). Після `GIN jsonb_path_ops` план став `Bitmap Index Scan → Bitmap Heap Scan`: індекс дав 3 334 кандидати, а cost знизився приблизно до 1 084 (~2 мс, у 5–7 разів швидше). Heap усе одно довелося прочитати повністю (`Heap Blocks: exact=1000`), бо картки розкидані по всій таблиці.

**Чому GIN не підходить для `->> 'device'`.** GIN індексує вміст документа для операторів `@>`, `?`, jsonpath, а `payload ->> 'device' = 'mobile'` — це рівність над результатом функції. Для нього GIN не використовується: лишився `Seq Scan` з помилковою оцінкою 250 рядків. Expression B-tree на `(payload ->> 'device')` дав `Bitmap Index Scan` і точну статистику (≈ 8 490 рядків).

**Read/write trade-off.** GIN пришвидшує вибіркові containment-запити, але займає 1 144 kB (≈ 14 % таблиці на 8 000 kB). Кожен INSERT або UPDATE payload оновлює багато записів GIN, тож запис дорожчає. Кожен expression-індекс додає ще одну структуру, яку треба підтримувати.

**JSONB чи normalized.** JSONB я обрала б для сирого event-log зі схемою, що часто змінюється, і для рідкісних ad-hoc запитів. Нормалізовану таблицю — для curated features і регулярної аналітики: в нашому прогоні вона читала 94 буфери замість 1 000 і мала cost 269 проти 1 957, плюс типи й `CHECK`.

**Новий тип `refund`.** У JSONB-підході достатньо почати писати `event_type = 'refund'` з payload `{order_id, amount, reason}`: міграція не потрібна, але варто додати валідацію (CHECK із `jsonb_typeof` або перевірку в застосунку) і, за потреби, expression-індекс на `order_id`. У нормалізованому підході потрібна міграція: `CREATE TABLE t7_hw_refunds_norm` з FK на checkout або order, `CHECK (amount > 0)`, індекси, backfill з events і оновлення ETL та feature-запитів.